# Chameleon setup notebook for proj08 Smart Transaction Categorization

Use this notebook in the Chameleon Jupyter environment to reserve the 3 `m1.large` VMs required by the Kubernetes deployment. The recommended path is:

1. Run the reservation/security-group cells here.
2. Use the printed reserved flavor in `proj08-iac/tf/kvm`.
3. Bootstrap Kubernetes with the repo Ansible/Kubespray playbooks.
4. Run `scripts/deploy-k8s-stack.sh` on node1.

The optional direct-server cell is left disabled by default because Terraform should own the final 3-node private network.

In [ ]:
from chi import context, lease, network, server
import chi
import datetime
import os
import time

PROJECT_SUFFIX = "proj08"
LEASE_NAME = "k8s-cluster-zy3180"
NODE_COUNT = 3
LEASE_HOURS = 168  # one week for the production-style traffic window
FLAVOR_NAME = "m1.large"
IMAGE_NAME = "CC-Ubuntu24.04"
KEY_NAME = "id_rsa_chameleon"

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

print({
    "lease_name": LEASE_NAME,
    "node_count": NODE_COUNT,
    "flavor": FLAVOR_NAME,
    "lease_hours": LEASE_HOURS,
})


## 1. Reserve three `m1.large` nodes

This creates the Chameleon lease. Terraform will use the reserved flavor id/name to create node1, node2, and node3 with the private network expected by Kubespray.

In [ ]:
l = lease.Lease(LEASE_NAME, duration=datetime.timedelta(hours=LEASE_HOURS))
l.add_flavor_reservation(id=chi.server.get_flavor_id(FLAVOR_NAME), amount=NODE_COUNT)
l.submit(idempotent=True)
l.show()

reserved_flavor = l.get_reserved_flavors()[0]
print("Reserved flavor name:", reserved_flavor.name)
print("Reserved flavor id:  ", reserved_flavor.id)
print("
Terraform example:")
print(f"cd proj08-iac/tf/kvm && terraform apply -var suffix={PROJECT_SUFFIX} -var reservation={reserved_flavor.id}")


## 2. Create browser-accessible security groups

Terraform now creates and attaches an equivalent security group automatically. This cell is still useful if you need to attach ports manually in Horizon or to an already-created node1 port.

In [ ]:
security_groups = [
    {"name": f"allow-ssh-{PROJECT_SUFFIX}", "port": 22, "description": "SSH"},
    {"name": f"allow-http-{PROJECT_SUFFIX}", "port": 80, "description": "HTTP/redirect"},
    {"name": f"allow-https-{PROJECT_SUFFIX}", "port": 443, "description": "HTTPS/ingress"},
    {"name": f"allow-actual-web-{PROJECT_SUFFIX}", "port": 3001, "description": "Actual dev web fallback"},
    {"name": f"allow-actual-sync-{PROJECT_SUFFIX}", "port": 5006, "description": "Actual sync/server fallback"},
    {"name": f"allow-serving-{PROJECT_SUFFIX}", "port": 8000, "description": "MLflow externalIP or SmartCat local fallback"},
    {"name": f"allow-jupyter-{PROJECT_SUFFIX}", "port": 8888, "description": "Jupyter troubleshooting"},
    {"name": f"allow-grafana-local-{PROJECT_SUFFIX}", "port": 3000, "description": "local Grafana fallback"},
    {"name": f"allow-prometheus-local-{PROJECT_SUFFIX}", "port": 9090, "description": "local Prometheus fallback"},
    {"name": f"allow-mlflow-local-{PROJECT_SUFFIX}", "port": 5000, "description": "local MLflow fallback"},
    {"name": f"allow-minio-api-{PROJECT_SUFFIX}", "port": 9000, "description": "MinIO API"},
    {"name": f"allow-minio-console-{PROJECT_SUFFIX}", "port": 9001, "description": "MinIO console"},
    {"name": f"allow-k8s-actual-legacy-{PROJECT_SUFFIX}", "port": 30080, "description": "legacy Actual NodePort"},
    {"name": f"allow-k8s-actual-{PROJECT_SUFFIX}", "port": 30083, "description": "Actual HTTP NodePort fallback"},
    {"name": f"allow-k8s-serving-{PROJECT_SUFFIX}", "port": 30090, "description": "SmartCat serving NodePort"},
    {"name": f"allow-k8s-grafana-{PROJECT_SUFFIX}", "port": 30300, "description": "Grafana NodePort"},
    {"name": f"allow-k8s-actual-https-{PROJECT_SUFFIX}", "port": 30443, "description": "Actual HTTPS NodePort"},
    {"name": f"allow-k8s-minio-{PROJECT_SUFFIX}", "port": 30901, "description": "optional MinIO NodePort"},
    {"name": f"allow-k8s-prometheus-{PROJECT_SUFFIX}", "port": 30909, "description": "Prometheus NodePort"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({"name": sg["name"], "description": sg["description"]})
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)

print("Security groups created or verified:")
for sg in security_groups:
    print(f"- {sg['name']}: tcp/{sg['port']} ({sg['description']})")


## 3. Optional: launch direct servers instead of Terraform

Leave `CREATE_DIRECT_SERVERS = False` for the final Kubernetes deployment. Turn it on only for quick debugging. The final project should use Terraform because it creates the private `192.168.1.0/24` network used by Kubespray.

In [ ]:
CREATE_DIRECT_SERVERS = False

if CREATE_DIRECT_SERVERS:
    reserved_flavor = l.get_reserved_flavors()[0]
    nodes = []
    for index in range(1, NODE_COUNT + 1):
        name = f"node{index}-mlops-{PROJECT_SUFFIX}"
        vm = server.Server(name, image_name=IMAGE_NAME, flavor_name=reserved_flavor.name)
        vm.submit(idempotent=True)
        vm.refresh()
        for sg in security_groups:
            try:
                vm.add_security_group(sg["name"])
            except Exception:
                pass
        nodes.append(vm)

    nodes[0].associate_floating_ip()
    time.sleep(5)
    for vm in nodes:
        vm.refresh()
        vm.show()
else:
    print("Direct server launch skipped. Use Terraform with the reserved flavor for the final 3-node cluster.")


## 4. URL checklist after deployment

After Terraform/Ansible creates the cluster and `scripts/deploy-k8s-stack.sh` finishes on node1, replace `<FLOATING_IP>` with node1's floating IP.

In [ ]:
FLOATING_IP = "<FLOATING_IP>"
service_urls = {
    "Actual Budget HTTPS": f"https://{FLOATING_IP}:30443",
    "Actual Budget HTTPS sslip": f"https://actual.{FLOATING_IP}.sslip.io:30443",
    "Actual Budget HTTP fallback": f"http://{FLOATING_IP}:30083",
    "SmartCat Serving docs": f"http://{FLOATING_IP}:30090/docs",
    "SmartCat monitor summary": f"http://{FLOATING_IP}:30090/monitor/summary",
    "Grafana": f"http://{FLOATING_IP}:30300",
    "Prometheus": f"http://{FLOATING_IP}:30909",
    "MLflow": f"http://{FLOATING_IP}:8000",
    "MinIO console": f"http://{FLOATING_IP}:9001",
}
for name, url in service_urls.items():
    print(f"{name:30s} {url}")

print("
Use the HTTPS Actual URL for the UI. Public HTTP will trigger Chrome SharedArrayBuffer errors.")
